# ML Enhancement: Promotion Effectiveness Analysis

> **Status: OPTIONAL / EXPERIMENTAL.** Results are observational associations,
> not causal estimates of incremental sales, ROI, or cannibalization.

Analyzes price elasticity and observed sales differences around promotions.

## Business Objective
- Describe product price sensitivity with transparent OLS evidence
- Compare promoted, pre-promotion, and post-promotion calendar periods
- Preserve exact promoted receipt-line attribution by store and promotion episode
- Identify candidates for controlled follow-up experiments

## Method
- **Price Elasticity**: Observational log-log OLS of quantity versus price
- **Observed Promotion Lift**: Calendar-day-normalized sales comparison
- **Post-Period Change**: Descriptive comparison after each promotion episode

## Data Flow
```
Configured Silver source tables
  --> configured Gold price elasticity output (product-level)
  --> configured Gold promotion lift output (store/episode/product-level)
```

## Usage
Run after historical data load. Do not interpret these outputs as proof that a
promotion caused a sales change; controlled experiments are required for causal claims.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from datetime import datetime, timezone
import os
import numpy as np
from typing import Tuple
import mlflow


In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name: str, default: str | None = None) -> str | None:
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="ag")
GOLD_DB = get_env("GOLD_DB", default="au")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="promotion_effectiveness")
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
RECEIPT_LINES_TABLE = get_env("RECEIPT_LINES_TABLE", default="fact_receipt_lines")
PRODUCTS_TABLE = get_env("PRODUCTS_TABLE", default="dim_products")
PROMOTIONS_TABLE = get_env("PROMOTIONS_TABLE", default="fact_promotions")
PROMO_LINES_TABLE = get_env("PROMO_LINES_TABLE", default="fact_promo_lines")
PRICE_ELASTICITY_TABLE = get_env("PRICE_ELASTICITY_TABLE", default="price_elasticity")
PROMOTION_LIFT_TABLE = get_env("PROMOTION_LIFT_TABLE", default="promotion_lift")
PRICE_ELASTICITY_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PRICE_ELASTICITY_TABLE}"
PROMOTION_LIFT_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{PROMOTION_LIFT_TABLE}"

MIN_OBSERVATIONS = int(get_env("MIN_OBSERVATIONS", default="30"))
BASELINE_WINDOW_DAYS = int(get_env("BASELINE_WINDOW_DAYS", default="30"))
PROMO_EPISODE_GAP_DAYS = int(get_env("PROMO_EPISODE_GAP_DAYS", default="7"))
TOP_N_PRODUCTS = int(get_env("TOP_N_PRODUCTS", default="100"))

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Source tables: {RECEIPTS_TABLE}, {RECEIPT_LINES_TABLE}, {PRODUCTS_TABLE}, {PROMOTIONS_TABLE}")
print(f"Promotion line mapping: {PROMO_LINES_TABLE} (optional, preferred)")
print(f"Output tables: {PRICE_ELASTICITY_TABLE_NAME}, {PROMOTION_LIFT_TABLE_NAME}")
print("Analysis design: observational; no causal or ROI claims")
print("Analysis Parameters:")
print(f"  MIN_OBSERVATIONS: {MIN_OBSERVATIONS}")
print(f"  BASELINE_WINDOW_DAYS: {BASELINE_WINDOW_DAYS}")
print(f"  PROMO_EPISODE_GAP_DAYS: {PROMO_EPISODE_GAP_DAYS}")
print(f"  TOP_N_PRODUCTS: {TOP_N_PRODUCTS}")

# Physical ML contract used by repository validation and the pre-write gate.
ML_SOURCE_TABLES = ('fact_receipts', 'fact_receipt_lines', 'dim_products', 'fact_promotions', 'fact_promo_lines')
ML_OUTPUT_CONTRACTS = {'price_elasticity': [('product_id', 'long', False),
                      ('product_name', 'string', False),
                      ('product_category', 'string', False),
                      ('elasticity_coefficient', 'double', True),
                      ('elasticity_category', 'string', False),
                      ('confidence_interval_lower', 'double', True),
                      ('confidence_interval_upper', 'double', True),
                      ('standard_error', 'double', True),
                      ('r_squared', 'double', True),
                      ('n_observations', 'long', False),
                      ('sxx', 'double', False),
                      ('sse', 'double', True),
                      ('avg_price', 'double', False),
                      ('regular_price', 'double', False),
                      ('avg_daily_quantity', 'double', False),
                      ('elasticity_evidence_status', 'string', False),
                      ('analysis_design', 'string', False),
                      ('computed_at', 'timestamp', False)],
 'promotion_lift': [('store_id', 'long', False),
                    ('promo_code', 'string', False),
                    ('promo_episode_id', 'string', False),
                    ('product_id', 'long', False),
                    ('product_name', 'string', False),
                    ('product_category', 'string', False),
                    ('discount_type', 'string', False),
                    ('avg_discount', 'double', True),
                    ('avg_discount_pct', 'double', True),
                    ('promo_start_date', 'date', False),
                    ('promo_end_date', 'date', False),
                    ('promoted_receipts_product', 'long', False),
                    ('promoted_line_count', 'long', False),
                    ('baseline_calendar_days', 'long', False),
                    ('promo_calendar_days', 'long', False),
                    ('post_calendar_days', 'long', False),
                    ('baseline_daily_quantity', 'double', False),
                    ('promo_daily_quantity', 'double', False),
                    ('post_daily_quantity', 'double', False),
                    ('total_promoted_quantity', 'double', False),
                    ('total_promoted_revenue', 'double', False),
                    ('observed_promo_lift_pct', 'double', True),
                    ('observed_post_period_change_pct', 'double', True),
                    ('observed_net_change_pct', 'double', True),
                    ('observed_lift_category', 'string', False),
                    ('analysis_design', 'string', False),
                    ('computed_at', 'timestamp', False)]}


In [ ]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def ensure_database(name: str) -> None:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")

def read_silver(table_name: str):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def save_gold(df, table_name: str) -> None:
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    df.write.format("delta").mode("overwrite").saveAsTable(full_name)
    print(f"  {full_name}: {df.count()} rows")

def silver_exists(table_name: str) -> bool:
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

def resolve_table_column(df, table_name: str, *candidates: str) -> str:
    columns_by_lower = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        resolved = columns_by_lower.get(candidate.lower())
        if resolved is not None:
            return resolved
    raise RuntimeError(
        f"Unable to resolve any of {candidates} in {LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}. Available columns: {df.columns}"
    )

ensure_database(GOLD_DB)

mlflow.set_experiment(EXPERIMENT_NAME)

def _normalize_ml_type(data_type):
    value = data_type.replace("bigint", "long").replace("integer", "int")
    return value


def validate_ml_output(frame, table_name):
    contract = ML_OUTPUT_CONTRACTS[table_name]
    expected = tuple((name, data_type) for name, data_type, _ in contract)
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"ML output schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )
    non_nullable = tuple(name for name, _, nullable in contract if not nullable)
    null_flags = frame.agg(
        *(F.max(F.col(name).isNull().cast("int")).alias(name) for name in non_nullable)
    ).first().asDict()
    null_columns = [name for name in non_nullable if null_flags.get(name)]
    if null_columns:
        raise RuntimeError(
            f"ML output {table_name} has nulls in required columns: {null_columns}"
        )
    return frame



---
## Part 1: Data Preparation

Join receipt lines with product dimensions and promotion data.

In [ ]:
print("="*60)
print("DATA PREPARATION")
print("="*60)

required_tables = [RECEIPTS_TABLE, RECEIPT_LINES_TABLE, PRODUCTS_TABLE, PROMOTIONS_TABLE]
missing_tables = [table for table in required_tables if not silver_exists(table)]
if missing_tables:
    raise RuntimeError(f"Missing required tables: {missing_tables}")
print("All required tables present.")

In [ ]:
# Load receipt headers so every line has an exact store attribution.
df_receipts_raw = read_silver(RECEIPTS_TABLE)
receipt_id_col = resolve_table_column(
    df_receipts_raw, RECEIPTS_TABLE, "receipt_id_ext", "ReceiptIdExt", "receipt_id", "ReceiptId"
)
receipt_store_col = resolve_table_column(df_receipts_raw, RECEIPTS_TABLE, "store_id", "StoreID")
receipt_event_ts_col = resolve_table_column(df_receipts_raw, RECEIPTS_TABLE, "event_ts", "EventTS")
receipt_type_col = resolve_table_column(df_receipts_raw, RECEIPTS_TABLE, "receipt_type", "ReceiptType")
df_receipts = (
    df_receipts_raw
    .select(
        F.col(receipt_id_col).alias("receipt_id_ext"),
        F.col(receipt_store_col).cast("long").alias("store_id"),
        F.col(receipt_event_ts_col).cast("timestamp").alias("receipt_event_ts"),
        F.col(receipt_type_col).alias("receipt_type"),
    )
    .filter(F.upper(F.col("receipt_type")) == "SALE")
    .dropDuplicates(["receipt_id_ext"])
)

df_receipt_lines = read_silver(RECEIPT_LINES_TABLE)
df_products_raw = read_silver(PRODUCTS_TABLE)
product_id_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_id", "id", "ID")
product_name_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_name", "ProductName")
product_category_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_category", "category", "Category")
product_subcategory_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_subcategory", "subcategory", "Subcategory")
product_regular_price_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "regular_price", "sale_price", "SalePrice")
product_cost_col = resolve_table_column(df_products_raw, PRODUCTS_TABLE, "product_cost", "cost", "Cost")
df_products = df_products_raw.select(
    F.col(product_id_col).cast("long").alias("product_id"),
    F.col(product_name_col).alias("product_name"),
    F.col(product_category_col).alias("product_category"),
    F.col(product_subcategory_col).alias("product_subcategory"),
    F.col(product_regular_price_col).cast("double").alias("regular_price"),
    F.col(product_cost_col).cast("double").alias("product_cost"),
)

df_promotions_raw = read_silver(PROMOTIONS_TABLE)
promotion_columns_by_lower = {column.lower(): column for column in df_promotions_raw.columns}
promotion_event_ts_col = resolve_table_column(df_promotions_raw, PROMOTIONS_TABLE, "event_ts", "EventTS")
promotion_receipt_id_col = resolve_table_column(
    df_promotions_raw, PROMOTIONS_TABLE, "receipt_id_ext", "receipt_id", "ReceiptIdExt", "ReceiptId"
)
promotion_store_id_col = resolve_table_column(df_promotions_raw, PROMOTIONS_TABLE, "store_id", "StoreID")
promotion_promo_code_col = resolve_table_column(df_promotions_raw, PROMOTIONS_TABLE, "promo_code", "PromoCode")
promotion_discount_type_col = resolve_table_column(
    df_promotions_raw, PROMOTIONS_TABLE, "discount_type", "DiscountType"
)
promotion_discount_cents_col = promotion_columns_by_lower.get("discount_cents") or promotion_columns_by_lower.get("discountcents")
promotion_discount_amount_col = promotion_columns_by_lower.get("discount_amount") or promotion_columns_by_lower.get("discountamount")
if promotion_discount_cents_col is None and promotion_discount_amount_col is None:
    raise RuntimeError(
        f"Unable to resolve discount amount columns in {LAKEHOUSE_NAME}.{SILVER_DB}.{PROMOTIONS_TABLE}. "
        f"Available columns: {df_promotions_raw.columns}"
    )
promotion_discount_expr = (
    F.col(promotion_discount_cents_col).cast("double") / F.lit(100.0)
    if promotion_discount_cents_col is not None
    else F.col(promotion_discount_amount_col).cast("double")
)
df_promotions = (
    df_promotions_raw
    .select(
        F.to_date(F.col(promotion_event_ts_col).cast("timestamp")).alias("promo_date"),
        F.col(promotion_receipt_id_col).alias("receipt_id"),
        F.col(promotion_store_id_col).cast("long").alias("store_id"),
        F.col(promotion_promo_code_col).alias("promo_code"),
        F.col(promotion_discount_type_col).alias("discount_type"),
        promotion_discount_expr.alias("discount_amount"),
    )
    .filter(F.col("receipt_id").isNotNull() & F.col("store_id").isNotNull())
    .dropDuplicates(["store_id", "receipt_id", "promo_code", "discount_type", "promo_date"])
    .cache()
)

print(f"Receipts: {df_receipts.count():,} rows")
print(f"Receipt lines: {df_receipt_lines.count():,} rows")
print(f"Products: {df_products.count():,} rows")
print(f"Promotions: {df_promotions.count():,} rows")

In [ ]:
# Enrich exact receipt lines with receipt-store and product attributes.
df_sales = (
    df_receipt_lines
    .join(df_receipts.select("receipt_id_ext", "store_id"), on="receipt_id_ext", how="inner")
    .join(df_products, on="product_id", how="inner")
    .select(
        F.col("store_id").cast("long").alias("store_id"),
        F.to_date(F.col("event_ts").cast("timestamp")).alias("date"),
        "receipt_id_ext",
        F.col("line_num").cast("long").alias("line_num"),
        F.col("product_id").cast("long").alias("product_id"),
        "product_name",
        "product_category",
        "product_subcategory",
        F.col("quantity").cast("double").alias("quantity"),
        F.when(
            F.col("quantity").cast("double") > 0,
            F.col("ext_cents").cast("double")
            / (F.col("quantity").cast("double") * F.lit(100.0)),
        ).alias("unit_price"),
        (F.col("ext_cents") / F.lit(100.0)).alias("ext_price"),
        "promo_code",
        "regular_price",
        "product_cost",
    )
    .withColumn("is_promoted", F.when(F.col("promo_code").isNotNull(), 1).otherwise(0))
    .withColumn(
        "discount_pct",
        F.when(
            (F.col("regular_price") > 0) & (F.col("unit_price") < F.col("regular_price")),
            ((F.col("regular_price") - F.col("unit_price")) / F.col("regular_price")) * 100,
        ).otherwise(0.0),
    )
    .cache()
)
print(f"\nEnriched exact sales lines: {df_sales.count():,} rows")
df_sales.printSchema()

---
## Part 2: Price Elasticity Estimation

Calculate price elasticity using log-log regression:
```
log(quantity) = β₀ + β₁ * log(price) + ε
```
Where β₁ is the price elasticity coefficient.

In [ ]:
print("="*60)
print("OBSERVATIONAL PRICE ELASTICITY ANALYSIS")
print("="*60)

df_top_products = (
    df_sales
    .groupBy("product_id", "product_name", "product_category")
    .agg(
        F.sum("ext_price").alias("total_revenue"),
        F.sum("quantity").alias("total_units"),
        F.countDistinct("date").alias("days_sold"),
    )
    .filter(F.col("days_sold") >= MIN_OBSERVATIONS)
    .orderBy(F.desc("total_revenue"))
    .limit(TOP_N_PRODUCTS)
)
print(f"\nAnalyzing top {TOP_N_PRODUCTS} products with >= {MIN_OBSERVATIONS} observed sales days")
df_top_products.show(10, truncate=False)

In [ ]:
# Aggregate sales by product and date for elasticity calculation
df_daily_sales = (
    df_sales
    .join(
        df_top_products.select("product_id"),
        on="product_id",
        how="inner"
    )
    .groupBy("product_id", "product_name", "product_category", "date")
    .agg(
        F.sum("quantity").alias("daily_quantity"),
        F.avg("unit_price").alias("avg_price"),
        F.max("regular_price").alias("regular_price"),
        F.max("is_promoted").alias("is_promoted")
    )
    .filter(
        (F.col("daily_quantity") > 0) &
        (F.col("avg_price") > 0)
    )
    .withColumn("log_quantity", F.log(F.col("daily_quantity")))
    .withColumn("log_price", F.log(F.col("avg_price")))
)

print(f"\nDaily sales aggregation: {df_daily_sales.count():,} product-day observations")

In [ ]:
# Calculate per-product means required for log-log OLS.
df_elasticity_stats = (
    df_daily_sales
    .groupBy("product_id", "product_name", "product_category")
    .agg(
        F.count("*").alias("n_observations"),
        F.avg("log_price").alias("mean_log_price"),
        F.avg("log_quantity").alias("mean_log_quantity"),
        F.avg("avg_price").alias("avg_price"),
        F.avg("regular_price").alias("regular_price"),
        F.avg("daily_quantity").alias("avg_daily_quantity"),
    )
)
print(f"\nElasticity statistics calculated for {df_elasticity_stats.count():,} products")

In [ ]:
# Compute OLS slope diagnostics. For simple OLS with an intercept:
# slope SE = sqrt((SSE / (n - 2)) / Sxx).
elasticity_join_keys = ["product_id", "product_name", "product_category"]
df_elasticity_calc = (
    df_daily_sales.alias("sales")
    .join(
        df_elasticity_stats.select(
            *elasticity_join_keys,
            F.col("n_observations").alias("stats_n_observations"),
            F.col("mean_log_price").alias("stats_mean_log_price"),
            F.col("mean_log_quantity").alias("stats_mean_log_quantity"),
            F.col("avg_price").alias("stats_avg_price"),
            F.col("regular_price").alias("stats_regular_price"),
            F.col("avg_daily_quantity").alias("stats_avg_daily_quantity"),
        ).alias("stats"),
        on=elasticity_join_keys,
        how="inner",
    )
    .withColumn("price_deviation", F.col("log_price") - F.col("stats_mean_log_price"))
    .withColumn("quantity_deviation", F.col("log_quantity") - F.col("stats_mean_log_quantity"))
    .withColumn("cross_product", F.col("price_deviation") * F.col("quantity_deviation"))
    .withColumn("price_sq_deviation", F.col("price_deviation") * F.col("price_deviation"))
    .withColumn("quantity_sq_deviation", F.col("quantity_deviation") * F.col("quantity_deviation"))
)

df_ols_sums = (
    df_elasticity_calc
    .groupBy(*elasticity_join_keys)
    .agg(
        F.first("stats_n_observations").alias("n_observations"),
        F.first("stats_avg_price").alias("avg_price"),
        F.first("stats_regular_price").alias("regular_price"),
        F.first("stats_avg_daily_quantity").alias("avg_daily_quantity"),
        F.sum("price_sq_deviation").alias("sxx"),
        F.sum("cross_product").alias("sxy"),
        F.sum("quantity_sq_deviation").alias("syy"),
    )
    .withColumn(
        "elasticity_coefficient",
        F.when(F.col("sxx") > 0, F.col("sxy") / F.col("sxx")).otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "sse",
        F.when(
            F.col("sxx") > 0,
            F.greatest(F.lit(0.0), F.col("syy") - (F.col("sxy") * F.col("sxy") / F.col("sxx"))),
        ).otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "standard_error",
        F.when(
            (F.col("n_observations") > 2) & (F.col("sxx") > 0),
            F.sqrt((F.col("sse") / (F.col("n_observations") - F.lit(2.0))) / F.col("sxx")),
        ).otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "r_squared",
        F.when(
            (F.col("sxx") > 0) & (F.col("syy") > 0),
            (F.col("sxy") * F.col("sxy")) / (F.col("sxx") * F.col("syy")),
        ).otherwise(F.lit(None).cast("double")),
    )
)

df_price_elasticity = (
    df_ols_sums
    .withColumn(
        "elasticity_category",
        F.when(F.col("elasticity_coefficient").isNull(), "Insufficient Evidence")
         .when(F.abs(F.col("elasticity_coefficient")) > 1.5, "Highly Elastic")
         .when(F.abs(F.col("elasticity_coefficient")) > 1.0, "Elastic")
         .when(F.abs(F.col("elasticity_coefficient")) > 0.5, "Unit Elastic")
         .otherwise("Inelastic"),
    )
    .withColumn(
        "confidence_interval_lower",
        F.col("elasticity_coefficient") - (F.lit(1.96) * F.col("standard_error")),
    )
    .withColumn(
        "confidence_interval_upper",
        F.col("elasticity_coefficient") + (F.lit(1.96) * F.col("standard_error")),
    )
    .withColumn(
        "elasticity_evidence_status",
        F.when(F.col("n_observations") < MIN_OBSERVATIONS, "INSUFFICIENT_SAMPLE")
         .when(F.col("sxx") <= 0, "INSUFFICIENT_PRICE_VARIATION")
         .when(F.col("standard_error").isNull(), "INVALID_STANDARD_ERROR")
         .otherwise("VALID_OBSERVATIONAL_ESTIMATE"),
    )
    .withColumn("analysis_design", F.lit("OBSERVATIONAL_LOG_LOG_OLS"))
    .withColumn("computed_at", F.current_timestamp())
    .select(
        "product_id", "product_name", "product_category", "elasticity_coefficient",
        "elasticity_category", "confidence_interval_lower", "confidence_interval_upper",
        "standard_error", "r_squared", "n_observations", "sxx", "sse", "avg_price",
        "regular_price", "avg_daily_quantity", "elasticity_evidence_status",
        "analysis_design", "computed_at",
    )
)
print(f"\nObservational price elasticity calculated for {df_price_elasticity.count():,} products")
print("\nSample results:")
df_price_elasticity.orderBy(F.abs(F.col("elasticity_coefficient")), ascending=False).show(10, truncate=False)

In [ ]:
# Save price elasticity to Gold layer
print(f"\nSaving {PRICE_ELASTICITY_TABLE_NAME}...")
df_price_elasticity = validate_ml_output(df_price_elasticity, "price_elasticity")
save_gold(df_price_elasticity, PRICE_ELASTICITY_TABLE)

---
## Part 3: Observed Promotion-Period Comparison

Compare exact promoted receipt lines with full eligible calendar periods:
- Baseline period before each store/promotion episode
- Exact promoted receipt lines during the episode
- Post-promotion period after the episode

These descriptive differences are not causal lift, ROI, or cannibalization estimates.

In [ ]:
print("="*60)
print("OBSERVED PROMOTION-PERIOD COMPARISON")
print("="*60)

promo_lines_available = silver_exists(PROMO_LINES_TABLE)
if promo_lines_available:
    df_promo_lines_raw = read_silver(PROMO_LINES_TABLE)
    promo_lines_receipt_id_col = resolve_table_column(
        df_promo_lines_raw, PROMO_LINES_TABLE, "receipt_id_ext", "receipt_id", "ReceiptIdExt", "ReceiptId"
    )
    promo_lines_promo_code_col = resolve_table_column(
        df_promo_lines_raw, PROMO_LINES_TABLE, "promo_code", "PromoCode"
    )
    promo_lines_line_number_col = resolve_table_column(
        df_promo_lines_raw, PROMO_LINES_TABLE, "line_number", "line_num", "LineNumber"
    )
    promo_lines_product_id_col = resolve_table_column(
        df_promo_lines_raw, PROMO_LINES_TABLE, "product_id", "ProductID"
    )
    df_promo_line_map = (
        df_promo_lines_raw
        .select(
            F.col(promo_lines_receipt_id_col).alias("receipt_id"),
            F.col(promo_lines_promo_code_col).alias("promo_code"),
            F.col(promo_lines_line_number_col).cast("long").alias("line_num"),
            F.col(promo_lines_product_id_col).cast("long").alias("product_id"),
        )
        .filter(
            F.col("receipt_id").isNotNull()
            & F.col("promo_code").isNotNull()
            & F.col("line_num").isNotNull()
            & F.col("product_id").isNotNull()
        )
        .join(
            df_receipts.select(F.col("receipt_id_ext").alias("receipt_id"), "store_id"),
            on="receipt_id", how="inner",
        )
        .distinct()
    )
    print(f"Using exact receipt-line mapping from {LAKEHOUSE_NAME}.{SILVER_DB}.{PROMO_LINES_TABLE}")
else:
    df_promo_line_map = (
        df_sales
        .select(
            "store_id", F.col("receipt_id_ext").alias("receipt_id"),
            "promo_code", "line_num", "product_id",
        )
        .filter(
            F.col("receipt_id").isNotNull()
            & F.col("promo_code").isNotNull()
            & F.col("line_num").isNotNull()
            & F.col("product_id").isNotNull()
        )
        .distinct()
    )
    print(f"{PROMO_LINES_TABLE} not found; using exact promo-coded receipt lines from sales")

df_promo_daily = (
    df_promotions
    .filter(
        F.col("promo_code").isNotNull()
        & F.col("discount_type").isNotNull()
        & F.col("promo_date").isNotNull()
    )
    .groupBy("store_id", "promo_code", "discount_type", "promo_date")
    .agg(F.countDistinct("receipt_id").alias("promoted_receipts"))
)

episode_order = Window.partitionBy("store_id", "promo_code", "discount_type").orderBy("promo_date")
episode_window = episode_order.rowsBetween(Window.unboundedPreceding, Window.currentRow)
df_promo_daily = (
    df_promo_daily
    .withColumn("prev_promo_date", F.lag("promo_date").over(episode_order))
    .withColumn("gap_days", F.datediff(F.col("promo_date"), F.col("prev_promo_date")))
    .withColumn(
        "episode_break",
        F.when(
            F.col("prev_promo_date").isNull() | (F.col("gap_days") > PROMO_EPISODE_GAP_DAYS), 1
        ).otherwise(0),
    )
    .withColumn("promo_episode_id", F.sum("episode_break").over(episode_window).cast("long"))
)

df_promo_events = (
    df_promotions.alias("promo")
    .join(
        df_promo_daily.select(
            "store_id", "promo_code", "discount_type", "promo_date", "promo_episode_id"
        ).alias("episode"),
        (F.col("promo.store_id") == F.col("episode.store_id"))
        & (F.col("promo.promo_code") == F.col("episode.promo_code"))
        & (F.col("promo.discount_type") == F.col("episode.discount_type"))
        & (F.col("promo.promo_date") == F.col("episode.promo_date")),
        how="inner",
    )
    .select(
        F.col("promo.store_id").alias("store_id"),
        F.col("promo.promo_code").alias("promo_code"),
        F.col("promo.discount_type").alias("discount_type"),
        F.col("promo.promo_date").alias("promo_date"),
        F.col("promo.receipt_id").alias("receipt_id"),
        F.col("promo.discount_amount").alias("discount_amount"),
        F.col("episode.promo_episode_id").alias("promo_episode_id"),
    )
)

df_promo_periods = (
    df_promo_events
    .groupBy("store_id", "promo_code", "discount_type", "promo_episode_id")
    .agg(
        F.min("promo_date").alias("promo_start_date"),
        F.max("promo_date").alias("promo_end_date"),
        F.countDistinct("receipt_id").alias("promoted_receipts"),
        F.avg("discount_amount").alias("avg_discount"),
    )
    .withColumn("baseline_start_date", F.date_sub(F.col("promo_start_date"), BASELINE_WINDOW_DAYS))
    .withColumn("baseline_end_date", F.date_sub(F.col("promo_start_date"), 1))
    .withColumn("post_promo_start_date", F.date_add(F.col("promo_end_date"), 1))
    .withColumn("post_promo_end_date", F.date_add(F.col("promo_end_date"), BASELINE_WINDOW_DAYS))
)
df_store_observation_bounds = (
    df_receipts
    .withColumn("receipt_date", F.to_date("receipt_event_ts"))
    .groupBy("store_id")
    .agg(
        F.min("receipt_date").alias("first_observed_date"),
        F.max("receipt_date").alias("last_observed_date"),
    )
)
sales_observation_bounds = df_store_observation_bounds.agg(
    F.min("first_observed_date").alias("first_observed_date"),
    F.max("last_observed_date").alias("last_observed_date"),
).first()
first_observed_date = sales_observation_bounds["first_observed_date"]
last_observed_date = sales_observation_bounds["last_observed_date"]
if first_observed_date is None or last_observed_date is None:
    raise RuntimeError("Promotion analysis has no observed sales dates.")
all_promo_period_count = df_promo_periods.count()
df_promo_periods = (
    df_promo_periods
    .join(df_store_observation_bounds, on="store_id", how="inner")
    .filter(
        (F.col("baseline_start_date") >= F.col("first_observed_date"))
        & (F.col("post_promo_end_date") <= F.col("last_observed_date"))
    )
    .drop("first_observed_date", "last_observed_date")
    .cache()
)
promo_period_count = df_promo_periods.count()
excluded_incomplete_periods = all_promo_period_count - promo_period_count
print(
    f"\nComplete-window promotion episodes: {promo_period_count:,}; "
    f"excluded outside each store's observed receipt window "
    f"(global {first_observed_date}..{last_observed_date}): "
    f"{excluded_incomplete_periods:,}"
)
df_promo_periods.show(10, truncate=False)

In [ ]:
# Resolve each promotion event to exact receipt lines, including store and episode.
df_episode_line_keys = (
    df_promo_events.alias("evt")
    .join(
        df_promo_line_map.alias("mapping"),
        (F.col("evt.store_id") == F.col("mapping.store_id"))
        & (F.col("evt.receipt_id") == F.col("mapping.receipt_id"))
        & (F.col("evt.promo_code") == F.col("mapping.promo_code")),
        how="inner",
    )
    .select(
        F.col("evt.store_id").alias("store_id"),
        F.col("evt.promo_code").alias("promo_code"),
        F.col("evt.discount_type").alias("discount_type"),
        F.col("evt.promo_episode_id").alias("promo_episode_id"),
        F.col("evt.promo_date").alias("promo_date"),
        F.col("evt.receipt_id").alias("receipt_id"),
        F.col("mapping.line_num").alias("line_num"),
        F.col("mapping.product_id").alias("product_id"),
    )
    .distinct()
)

df_promoted_lines_exact = (
    df_episode_line_keys.alias("promo")
    .join(
        df_sales.alias("sales"),
        (F.col("promo.store_id") == F.col("sales.store_id"))
        & (F.col("promo.receipt_id") == F.col("sales.receipt_id_ext"))
        & (F.col("promo.promo_code") == F.col("sales.promo_code"))
        & (F.col("promo.line_num") == F.col("sales.line_num"))
        & (F.col("promo.product_id") == F.col("sales.product_id"))
        & (F.col("promo.promo_date") == F.col("sales.date")),
        how="inner",
    )
    .select(
        F.col("promo.store_id").alias("store_id"),
        F.col("promo.promo_code").alias("promo_code"),
        F.col("promo.discount_type").alias("discount_type"),
        F.col("promo.promo_episode_id").alias("promo_episode_id"),
        F.col("promo.promo_date").alias("date"),
        F.col("promo.receipt_id").alias("receipt_id"),
        F.col("promo.line_num").alias("line_num"),
        F.col("promo.product_id").alias("product_id"),
        F.col("sales.quantity").alias("quantity"),
        F.col("sales.ext_price").alias("revenue"),
        F.col("sales.discount_pct").alias("discount_pct"),
    )
    .dropDuplicates([
        "store_id", "promo_code", "discount_type", "promo_episode_id",
        "receipt_id", "line_num", "product_id"
    ])
    .cache()
)

episode_key_cols = ["store_id", "promo_code", "discount_type", "promo_episode_id", "product_id"]
df_episode_products = (
    df_promoted_lines_exact
    .groupBy(*episode_key_cols)
    .agg(
        F.countDistinct("receipt_id").alias("promoted_receipts_product"),
        F.countDistinct(F.struct("receipt_id", "line_num")).alias("promoted_line_count"),
    )
    .join(
        df_promo_periods,
        on=["store_id", "promo_code", "discount_type", "promo_episode_id"], how="inner",
    )
    .cache()
)

df_sales_daily_all = (
    df_sales.groupBy("store_id", "product_id", "date")
    .agg(F.sum("quantity").alias("daily_quantity"), F.sum("ext_price").alias("daily_revenue"))
    .cache()
)
df_promoted_daily_exact = (
    df_promoted_lines_exact.groupBy(*episode_key_cols, "date")
    .agg(F.sum("quantity").alias("daily_quantity"), F.sum("revenue").alias("daily_revenue"))
    .cache()
)
df_product_promo_dates = (
    df_promoted_lines_exact.select("store_id", "product_id", "date").distinct().cache()
)
episode_product_count = df_episode_products.count()
print(f"\nExact store/episode/product combinations identified: {episode_product_count:,}")

In [ ]:
# Use an explicit calendar so zero-sales days remain in every denominator.
def aggregate_calendar_window_sales(
    start_col: str,
    end_col: str,
    prefix: str,
    daily_sales,
    daily_join_keys: list[str],
    exclude_promoted_days: bool = False,
):
    calendar = (
        df_episode_products
        .select(*episode_key_cols, start_col, end_col)
        .withColumn(
            "date",
            F.explode(F.sequence(F.col(start_col), F.col(end_col), F.expr("INTERVAL 1 DAY"))),
        )
        .drop(start_col, end_col)
    )
    if exclude_promoted_days:
        calendar = calendar.join(
            df_product_promo_dates,
            on=["store_id", "product_id", "date"], how="left_anti",
        )
    return (
        calendar
        .join(daily_sales, on=daily_join_keys, how="left")
        .groupBy(*episode_key_cols)
        .agg(
            F.sum(F.coalesce(F.col("daily_quantity"), F.lit(0.0))).alias(f"{prefix}_quantity"),
            F.sum(F.coalesce(F.col("daily_revenue"), F.lit(0.0))).alias(f"{prefix}_revenue"),
            F.count(F.lit(1)).alias(f"{prefix}_calendar_days"),
        )
        .withColumn(
            f"{prefix}_daily_quantity",
            F.when(
                F.col(f"{prefix}_calendar_days") > 0,
                F.col(f"{prefix}_quantity") / F.col(f"{prefix}_calendar_days"),
            ).otherwise(F.lit(0.0)),
        )
    )

df_baseline_sales = aggregate_calendar_window_sales(
    start_col="baseline_start_date",
    end_col="baseline_end_date",
    prefix="baseline",
    daily_sales=df_sales_daily_all,
    daily_join_keys=["store_id", "product_id", "date"],
    exclude_promoted_days=True,
)
print(f"\nBaseline calendar comparisons: {df_baseline_sales.count():,} combinations")

In [ ]:
# Exact promoted-line totals use the complete episode calendar denominator.
df_promoted_sales = aggregate_calendar_window_sales(
    start_col="promo_start_date",
    end_col="promo_end_date",
    prefix="promo",
    daily_sales=df_promoted_daily_exact,
    daily_join_keys=episode_key_cols + ["date"],
    exclude_promoted_days=False,
)
print(f"\nPromoted calendar comparisons: {df_promoted_sales.count():,} combinations")

df_post_promo_sales = aggregate_calendar_window_sales(
    start_col="post_promo_start_date",
    end_col="post_promo_end_date",
    prefix="post",
    daily_sales=df_sales_daily_all,
    daily_join_keys=["store_id", "product_id", "date"],
    exclude_promoted_days=True,
)
print(f"\nPost-promotion calendar comparisons: {df_post_promo_sales.count():,} combinations")

df_episode_discount_pct = (
    df_promoted_lines_exact.groupBy(*episode_key_cols)
    .agg(F.avg("discount_pct").alias("avg_discount_pct"))
)
print(f"Exact promoted-line discount statistics: {df_episode_discount_pct.count():,} combinations")

In [ ]:
# Combine descriptive period metrics without causal or ROI labels.
df_promotion_lift = (
    df_episode_products
    .join(df_promoted_sales, on=episode_key_cols, how="left")
    .join(df_baseline_sales, on=episode_key_cols, how="left")
    .join(df_post_promo_sales, on=episode_key_cols, how="left")
    .join(df_episode_discount_pct, on=episode_key_cols, how="left")
    .join(
        df_products.select("product_id", "product_name", "product_category"),
        on="product_id", how="inner",
    )
    .select(
        "store_id", "promo_code", "promo_episode_id", "product_id", "product_name",
        "product_category", "discount_type", "avg_discount", "avg_discount_pct",
        "promo_start_date", "promo_end_date", "promoted_receipts_product",
        "promoted_line_count",
        F.coalesce(F.col("baseline_calendar_days"), F.lit(0)).alias("baseline_calendar_days"),
        F.coalesce(F.col("promo_calendar_days"), F.lit(0)).alias("promo_calendar_days"),
        F.coalesce(F.col("post_calendar_days"), F.lit(0)).alias("post_calendar_days"),
        F.coalesce(F.col("baseline_daily_quantity"), F.lit(0.0)).alias("baseline_daily_quantity"),
        F.coalesce(F.col("promo_daily_quantity"), F.lit(0.0)).alias("promo_daily_quantity"),
        F.coalesce(F.col("post_daily_quantity"), F.lit(0.0)).alias("post_daily_quantity"),
        F.coalesce(F.col("promo_quantity"), F.lit(0.0)).alias("total_promoted_quantity"),
        F.coalesce(F.col("promo_revenue"), F.lit(0.0)).alias("total_promoted_revenue"),
    )
    .withColumn(
        "observed_promo_lift_pct",
        F.when(
            F.col("baseline_daily_quantity") > 0,
            ((F.col("promo_daily_quantity") - F.col("baseline_daily_quantity")) /
             F.col("baseline_daily_quantity")) * 100,
        ).otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "observed_post_period_change_pct",
        F.when(
            F.col("baseline_daily_quantity") > 0,
            ((F.col("post_daily_quantity") - F.col("baseline_daily_quantity")) /
             F.col("baseline_daily_quantity")) * 100,
        ).otherwise(F.lit(None).cast("double")),
    )
    .withColumn(
        "observed_net_change_pct",
        F.col("observed_promo_lift_pct") + F.col("observed_post_period_change_pct"),
    )
    .withColumn(
        "observed_lift_category",
        F.when(F.col("observed_promo_lift_pct") > 50, "MUCH_HIGHER_DURING_PROMOTION")
         .when(F.col("observed_promo_lift_pct") > 20, "HIGHER_DURING_PROMOTION")
         .when(F.col("observed_promo_lift_pct") > 0, "SLIGHTLY_HIGHER_DURING_PROMOTION")
         .otherwise("NOT_HIGHER_DURING_PROMOTION"),
    )
    .withColumn("analysis_design", F.lit("OBSERVATIONAL_CALENDAR_COMPARISON"))
    .withColumn("computed_at", F.current_timestamp())
)
print(f"\nObserved promotion-period analysis complete: {df_promotion_lift.count():,} combinations")
print("\nSample results:")
df_promotion_lift.orderBy(F.desc("observed_promo_lift_pct")).show(10, truncate=False)

In [ ]:
# Save observed promotion-period comparisons to Gold.
print(f"\nSaving {PROMOTION_LIFT_TABLE_NAME}...")
df_promotion_lift = df_promotion_lift.withColumn(
    "promo_episode_id",
    F.col("promo_episode_id").cast("string"),
)
df_promotion_lift = validate_ml_output(df_promotion_lift, "promotion_lift")
save_gold(df_promotion_lift, PROMOTION_LIFT_TABLE)

---
## Summary & Insights

In [ ]:
# Log observational analysis results to MLflow.
with mlflow.start_run(
    run_name=f"promo_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "observational_log_log_ols_and_calendar_comparison",
        "analysis_design": "observational_not_causal",
        "min_observations": MIN_OBSERVATIONS,
        "baseline_window_days": BASELINE_WINDOW_DAYS,
        "promo_episode_gap_days": PROMO_EPISODE_GAP_DAYS,
        "promotion_mapping_source": PROMO_LINES_TABLE if promo_lines_available else "exact_receipt_line_fallback",
        "top_n_products": TOP_N_PRODUCTS,
    })
    elasticity_count = spark.table(PRICE_ELASTICITY_TABLE_NAME).count()
    lift_count = spark.table(PROMOTION_LIFT_TABLE_NAME).count()
    mlflow.log_metrics({
        "elasticity_products": elasticity_count,
        "observed_promotion_records": lift_count,
    })
    print(f"MLflow run: {run.info.run_id}")
    print(f"Elasticity: {elasticity_count} products, observed promotion comparisons: {lift_count}")

In [ ]:
print("\n" + "="*60)
print("OBSERVATIONAL PROMOTION ANALYSIS COMPLETE")
print("="*60)

print("\n--- Price Elasticity Evidence ---")
df_price_elasticity.groupBy("elasticity_evidence_status").count().orderBy(F.desc("count")).show()

print("\nMost negative observed elasticity coefficients:")
df_price_elasticity.orderBy(F.asc("elasticity_coefficient")).select(
    "product_name", "product_category", "elasticity_coefficient",
    "confidence_interval_lower", "confidence_interval_upper", "n_observations"
).show(5, truncate=False)

print("\n--- Observed Promotion-Period Differences ---")
df_promotion_lift.groupBy("observed_lift_category").count().orderBy(F.desc("count")).show()

print("\nLargest observed during-promotion differences:")
df_promotion_lift.orderBy(F.desc("observed_promo_lift_pct")).select(
    "store_id", "promo_code", "promo_episode_id", "product_name",
    "observed_promo_lift_pct", "observed_post_period_change_pct",
    "observed_lift_category"
).show(5, truncate=False)

print("\nGold tables created:")
print(f"  - {PRICE_ELASTICITY_TABLE_NAME}")
print(f"  - {PROMOTION_LIFT_TABLE_NAME}")
print("\nThese observational outputs identify hypotheses; they do not establish causal lift or ROI.")